In [2]:
from pathlib import Path
import pickle
import re
import numpy as np
import pandas as pd
DATA_ROOT = Path(r"D:\SemanticBiods")
OUT_ROOT = Path(r"D:\SemanticBiods\data")
VECTORS = {"English": "EnglishVec", "French": "FrenchVec"}
RUNS = [
    ("main", 1800, 1990, 0),
    ("content", 1800, 1990, 500), 
    ("window", 1850, 1990, 0),
]
TOP_K = 10_000
VOCAB_SIZE = 100_000


def load_vocab(path):
    """Read a HistWords *-vocab.pkl as a list of str, row order preserved."""
    with open(path, "rb") as fh:
        try:
            vocab = pickle.load(fh)
        except Exception:
            fh.seek(0)
            vocab = pickle.load(fh, encoding="latin1")

    if isinstance(vocab, dict):
        vocab = [w for w, _ in sorted(vocab.items(), key=lambda kv: kv[1])]

    out = []
    for w in vocab:
        if isinstance(w, bytes):
            try:
                w = w.decode("utf-8")
            except Exception:
                w = w.decode("latin1")
        out.append(str(w))
    return out


def decades(base, start, end):
    pat = re.compile(r"^(\d{4})-vocab\.pkl$")
    years = []
    for p in base.glob("*-vocab.pkl"):
        m = pat.match(p.name)
        if m and start <= int(m.group(1)) <= end:
            year = int(m.group(1))
            if (base / f"{year}-w.npy").exists():
                years.append(year)
    return sorted(years)


def common_vocabulary(base, years, top_k, min_rank):
    keep = None
    ranks = {}

    for year in years:
        vocab = load_vocab(base / f"{year}-vocab.pkl")
        W = np.load(base / f"{year}-w.npy", mmap_mode="r")
        k = min(top_k, len(vocab))
        nonzero = np.linalg.norm(np.asarray(W[:k]), axis=1) > 0

        ranks[year] = {vocab[i]: i + 1
                       for i in range(min_rank, k) if nonzero[i]}
        present = set(ranks[year])
        keep = present if keep is None else (keep & present)
        print(f"    {year}: {len(present):6d} usable types, "
              f"{len(keep):6d} still common")

    words = sorted(keep, key=lambda w: np.mean([ranks[y][w] for y in years]))
    table = pd.DataFrame({y: [ranks[y][w] for w in words] for y in years},
                         index=pd.Index(words, name="word"))
    return table


def main():
    for language, folder in VECTORS.items():
        base = DATA_ROOT / folder
        for run, start, end, min_rank in RUNS:
            print(f"[{language} / {run}] {start}-{end}, ranks > {min_rank}")
            years = decades(base, start, end)
            table = common_vocabulary(base, years, TOP_K, min_rank)

            out = OUT_ROOT / language / run
            out.mkdir(parents=True, exist_ok=True)
            table.reset_index().to_csv(out / "common_vocabulary.csv",
                                       index=False)
            print(f"  -> {len(table)} words over {len(years)} decades\n")


if __name__ == "__main__":
    main()


[English / main] 1800-1990, ranks > 0
    1800:  10000 usable types,  10000 still common
    1810:  10000 usable types,   9590 still common
    1820:  10000 usable types,   9348 still common
    1830:  10000 usable types,   8892 still common
    1840:  10000 usable types,   8738 still common
    1850:  10000 usable types,   8593 still common
    1860:  10000 usable types,   8378 still common
    1870:  10000 usable types,   8206 still common
    1880:  10000 usable types,   8047 still common
    1890:  10000 usable types,   7842 still common
    1900:  10000 usable types,   7684 still common
    1910:  10000 usable types,   7300 still common
    1920:  10000 usable types,   7186 still common
    1930:  10000 usable types,   7027 still common
    1940:  10000 usable types,   6761 still common
    1950:  10000 usable types,   6679 still common
    1960:  10000 usable types,   6551 still common
    1970:  10000 usable types,   6301 still common
    1980:  10000 usable types,   6085 still 

In [1]:
from pathlib import Path
import pickle
import time
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
DATA_ROOT = Path(r"D:\SemanticBiods")
OUT_ROOT = Path(r"D:\SemanticBiods\data")
VECTORS = {"English": "EnglishVec", "French": "FrenchVec"}
RUNS = [
    ("main", 1800, 1990, 0),
    ("content", 1800, 1990, 500),
    ("window", 1850, 1990, 0),
]
TOP_LOCAL = 50 
TOP_TURNOVER = 100
VOCAB_SIZE = 100_000
CHUNK = 1024


def load_vocab(path):
    with open(path, "rb") as fh:
        try:
            vocab = pickle.load(fh)
        except Exception:
            fh.seek(0)
            vocab = pickle.load(fh, encoding="latin1")
    if isinstance(vocab, dict):
        vocab = [w for w, _ in sorted(vocab.items(), key=lambda kv: kv[1])]
    out = []
    for w in vocab:
        if isinstance(w, bytes):
            try:
                w = w.decode("utf-8")
            except Exception:
                w = w.decode("latin1")
        out.append(str(w))
    return out


def decade_vectors(base, year, words):
    """L2-normalised embedding rows for `words`, in the given order."""
    vocab = load_vocab(base / f"{year}-vocab.pkl")
    index = {w: i for i, w in enumerate(vocab)}
    rows = np.fromiter((index[w] for w in words), np.int64, len(words))

    W = np.load(base / f"{year}-w.npy", mmap_mode="r")
    V = np.asarray(W[rows], dtype=np.float32)
    norm = np.linalg.norm(V, axis=1, keepdims=True)
    norm[norm == 0] = 1.0
    return V / norm


def similarity_matrix(V):
    """Cosine similarity with the diagonal zeroed out."""
    S = (V @ V.T).astype(np.float32)
    np.clip(S, -1.0, 1.0, out=S)
    np.fill_diagonal(S, 0.0)
    return S


def cohesion_and_neighbours(S, top_local, top_turnover):
    n = S.shape[0]
    glob = (S.sum(axis=1, dtype=np.float64) / (n - 1)).astype(np.float32)

    local = np.empty(n, dtype=np.float32)
    top = np.empty((n, top_turnover), dtype=np.int32)

    for a in range(0, n, CHUNK):
        b = min(a + CHUNK, n)
        block = S[a:b].astype(np.float32, copy=True)
        block[np.arange(b - a), np.arange(a, b)] = -np.inf   # never pick self

        idx = np.argpartition(block, n - top_turnover, axis=1)[:, -top_turnover:]
        val = np.take_along_axis(block, idx, axis=1)
        order = np.argsort(-val, axis=1)

        top[a:b] = np.take_along_axis(idx, order, axis=1)
        local[a:b] = np.take_along_axis(val, order, axis=1)[:, :top_local].mean(1)

    return glob, local, top


def relational_change(S1, S2):
    """1 - Pearson r between the two similarity profiles of each word.

    Both diagonals are exactly 0, so they add nothing to the sums, the sums of
    squares or the cross-product; the profile length is simply n - 1.
    """
    n = S1.shape[0]
    p = n - 1
    r = np.empty(n, dtype=np.float64)

    for a in range(0, n, CHUNK):
        b = min(a + CHUNK, n)
        A = S1[a:b].astype(np.float64)
        B = S2[a:b].astype(np.float64)

        sa, sb = A.sum(1), B.sum(1)
        cov = (A * B).sum(1) - sa * sb / p
        va = (A * A).sum(1) - sa * sa / p
        vb = (B * B).sum(1) - sb * sb / p

        den = np.sqrt(np.maximum(va, 0) * np.maximum(vb, 0))
        r[a:b] = np.where(den > 0, cov / np.where(den > 0, den, 1.0), np.nan)

    return (1.0 - np.clip(r, -1.0, 1.0)).astype(np.float32)


def neighbour_turnover(prev_top, curr_top, k):
    n = prev_top.shape[0]

    def indicator(top):
        rows = np.repeat(np.arange(n), top.shape[1])
        data = np.ones(rows.size, dtype=np.int8)
        return csr_matrix((data, (rows, top.ravel())), shape=(n, n),
                          dtype=np.int8)

    overlap = np.asarray(
        indicator(prev_top).multiply(indicator(curr_top)).sum(axis=1)).ravel()
    return (1.0 - overlap / k).astype(np.float32)


def main():
    for language, folder in VECTORS.items():
        base = DATA_ROOT / folder
        for run, start, end, _ in RUNS:
            out = OUT_ROOT / language / run
            ranks = pd.read_csv(out / "common_vocabulary.csv")
            words = ranks["word"].astype(str).tolist()
            years = [int(c) for c in ranks.columns if c != "word"]
            print(f"[{language} / {run}] {len(words)} words, "
                  f"{len(years)} decades")

            t0 = time.time()
            features, changes = [], []
            prev_S = prev_top = prev_year = None

            for year in years:
                V = decade_vectors(base, year, words)
                S = similarity_matrix(V)
                glob, local, top = cohesion_and_neighbours(
                    S, TOP_LOCAL, TOP_TURNOVER)

                rank = ranks[str(year)].to_numpy(dtype=np.float64)
                features.append(pd.DataFrame({
                    "year": year,
                    "word": words,
                    "rank": rank.astype(int),
                    # Zipfian proxy for log frequency; larger = more frequent
                    "Frequency": -np.log(rank / VOCAB_SIZE),
                    "Global": glob,
                    "Local": local,
                }))

                if prev_S is not None:
                    changes.append(pd.DataFrame({
                        "transition_start": prev_year,
                        "transition_end": year,
                        "word": words,
                        "RSC": relational_change(prev_S, S),
                        "Turnover": neighbour_turnover(prev_top, top,
                                                       TOP_TURNOVER),
                    }))

                prev_S, prev_top, prev_year = S, top, year
                print(f"    {year} done", flush=True)

            pd.concat(features, ignore_index=True).to_csv(
                out / "decade_features.csv.gz", index=False, compression="gzip")
            pd.concat(changes, ignore_index=True).to_csv(
                out / "decade_change.csv.gz", index=False, compression="gzip")
            print(f"  -> {(time.time() - t0) / 60:.1f} min\n", flush=True)


if __name__ == "__main__":
    main()


[English / main] 5963 words, 20 decades
    1800 done
    1810 done
    1820 done
    1830 done
    1840 done
    1850 done
    1860 done
    1870 done
    1880 done
    1890 done
    1900 done
    1910 done
    1920 done
    1930 done
    1940 done
    1950 done
    1960 done
    1970 done
    1980 done
    1990 done
  -> 0.4 min

[English / content] 5239 words, 20 decades
    1800 done
    1810 done
    1820 done
    1830 done
    1840 done
    1850 done
    1860 done
    1870 done
    1880 done
    1890 done
    1900 done
    1910 done
    1920 done
    1930 done
    1940 done
    1950 done
    1960 done
    1970 done
    1980 done
    1990 done
  -> 0.4 min

[English / window] 6533 words, 15 decades
    1850 done
    1860 done
    1870 done
    1880 done
    1890 done
    1900 done
    1910 done
    1920 done
    1930 done
    1940 done
    1950 done
    1960 done
    1970 done
    1980 done
    1990 done
  -> 0.4 min

[French / main] 6057 words, 20 decades
    1800 done
    1810 d

In [1]:
from pathlib import Path
import pandas as pd
OUT_ROOT = Path(r"D:\SemanticBiods\data")
LANGUAGES = ["English", "French"]
RUNS = ["main", "content", "window"]
OUTCOMES = ["RSC", "Turnover"]
STATIC = ["Frequency", "Global", "Local"]
LAGS = [0, 1, 2, 3, 4, 5]
STEP = 10 


def build(features, changes, outcome):
    df = changes[["transition_start", "transition_end", "word", outcome]]
    df = df.rename(columns={outcome: "Outcome"}).copy()

    for lag in LAGS:
        block = features[["year", "word"] + STATIC].copy()
        block["transition_start"] = block["year"] + lag * STEP
        block = block.drop(columns="year").rename(
            columns={f: f"{f}_lag{lag}" for f in STATIC})
        df = df.merge(block, on=["word", "transition_start"], how="left",
                      validate="one_to_one")

    past = changes[["transition_end", "word", outcome]].rename(
        columns={"transition_end": "year", outcome: "PastChange"})
    for lag in LAGS:
        block = past.copy()
        block["transition_start"] = block["year"] + lag * STEP
        block = block[["word", "transition_start", "PastChange"]].rename(
            columns={"PastChange": f"PastChange_lag{lag}"})
        df = df.merge(block, on=["word", "transition_start"], how="left",
                      validate="one_to_one")

    cols = [f"{f}_lag{l}" for f in STATIC + ["PastChange"] for l in LAGS]
    df = df.dropna(subset=["Outcome"] + cols)
    return df.sort_values(["transition_start", "word"]).reset_index(drop=True)


def main():
    rows = []
    for language in LANGUAGES:
        for run in RUNS:
            out = OUT_ROOT / language / run
            features = pd.read_csv(out / "decade_features.csv.gz")
            changes = pd.read_csv(out / "decade_change.csv.gz")

            for outcome in OUTCOMES:
                df = build(features, changes, outcome)
                df.to_csv(out / f"lagged_{outcome}.csv.gz", index=False,
                          compression="gzip")
                rows.append({
                    "language": language, "run": run, "outcome": outcome,
                    "words": df["word"].nunique(),
                    "transitions": df["transition_start"].nunique(),
                    "first_transition": int(df["transition_start"].min()),
                    "last_transition": int(df["transition_start"].max()),
                    "observations": len(df),
                })
                print(f"[{language} / {run}] {outcome}: {len(df)} rows, "
                      f"{rows[-1]['transitions']} transitions", flush=True)

    summary = pd.DataFrame(rows)
    (OUT_ROOT / "tables").mkdir(parents=True, exist_ok=True)
    summary.to_csv(OUT_ROOT / "tables" / "design_summary.csv", index=False)
    print("\n", summary.to_string(index=False), sep="")


if __name__ == "__main__":
    main()


[English / main] RSC: 77519 rows, 13 transitions
[English / main] Turnover: 77519 rows, 13 transitions
[English / content] RSC: 68107 rows, 13 transitions
[English / content] Turnover: 68107 rows, 13 transitions
[English / window] RSC: 52264 rows, 8 transitions
[English / window] Turnover: 52264 rows, 8 transitions
[French / main] RSC: 78741 rows, 13 transitions
[French / main] Turnover: 78741 rows, 13 transitions
[French / content] RSC: 68731 rows, 13 transitions
[French / content] Turnover: 68731 rows, 13 transitions
[French / window] RSC: 54576 rows, 8 transitions
[French / window] Turnover: 54576 rows, 8 transitions

language     run  outcome  words  transitions  first_transition  last_transition  observations
 English    main      RSC   5963           13              1860             1980         77519
 English    main Turnover   5963           13              1860             1980         77519
 English content      RSC   5239           13              1860             1980      